In [0]:
%sql
Select current_catalog() ,  current_database() , current_schema()
--schema and database are same in databricks.

In [0]:
%sql
LIST '/Volumes/pyspark_practice/default/files'
--i have created  one volume because i need to save file 
-- i just cretaed volume , inside catlog insde a schema , 
--but if you notice ; volume/catlog/db/folder/files

In [0]:
%sql
Select * from csv.`/Volumes/pyspark_practice/default/files/Practice/BigMart Sales.csv` Limit 10
-- pay atention , it is not single quotes : it is bakcticks  not ' '  ; instead ``
-- Also , 1st row is not header ,

In [0]:
%sql
-- 
Select * from read_files('/Volumes/pyspark_practice/default/files/Practice/BigMart Sales.csv',
format =>'csv')
limit 10 ;


In [0]:
df =  (spark.read
      .format("csv")
      .option("header", "true")
      .option("inferSchema", "true")
      .load("/Volumes/pyspark_practice/default/files/Practice/BigMart Sales.csv"))

display(df)


there is one more way to write same code , 
insead of using () , we are using \ 
and instead of display(df)
we are using df.display()


In [0]:
df =  spark.read\
      .format("csv")\
      .option("header", "true")\
      .option("inferSchema", "true")\
      .load("/Volumes/pyspark_practice/default/files/Practice/BigMart Sales.csv")

df.display(10)

In [0]:
%sql
-- CTAS
drop table if exists BigMart_sales ;
-- ; is mandataory before create statement

--above we have one select statement  , just as create table <tablename as : 
Create table BigMart_sales as 
Select * from read_files('/Volumes/pyspark_practice/default/files/Practice/BigMart Sales.csv',
format =>'csv')


In [0]:
%sql
select * from BigMart_sales limit 10 ;

In [0]:
%sql
describe  table  BigMart_sales

In [0]:
%sql
Describe Table extended BigMart_sales

Managed vs External Tables in Databricks

Managed Tables

. Databricks manages both the data and metadata.
. Data is stored within Databricks' managed storage.
. Dropping the table also deletes the data.
. Recommended for creating new tables.

External Tables

. Databricks only manages the table metadata.
. Dropping the table does not delete the data.
· Supports multiple formats, including Delta Lake.
. Ideal for sharing data across platforms or using existing external data.

In [0]:
# python version of CTAS . 
df_python = (spark.read.format("csv")
             .option("header","true")
             .load("/Volumes/pyspark_practice/default/files/Practice/BigMart Sales.csv"))
#df_python.display(10)
df_python.write\
    .mode("overWrite")\
    .saveAsTable("workspace.default.big_mart")

udt  =  spark.table("workspace.default.big_mart")
udt.display(5)



### 1. **What does `write.saveAsTable()` do?**

```python
df_python.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.big_mart")
```

*   This writes the **DataFrame** `df_python` into a **managed table** in Spark’s catalog (in this case, `workspace.default.big_mart`).
*   After this, the data is stored in the underlying storage (e.g., Delta, Parquet) and registered in the **Hive metastore** or Spark catalog.
*   The `mode("overwrite")` ensures that if the table already exists, it will be replaced.

***

### 2. **What does `spark.table("workspace.default.big_mart")` do?**

*   This **reads** the table from the Spark catalog back into a DataFrame.
*   It’s equivalent to:

```python
SELECT * FROM workspace.default.big_mart
```

*   So, `spark.table()` is **not writing anything**; it’s just a convenient way to load the table you saved earlier.

***

### 3. **Why do we need `spark.table()` then?**

*   After writing, if you want to **reuse the data** in your Spark session (or validate what was written), you need to read it back.
*   Example:

```python
udt = spark.table("workspace.default.big_mart")
udt.show()
```

This confirms the data is persisted and accessible as a table.

***

### ✅ Key Point:

*   `write.saveAsTable()` → **Writes** DataFrame to a table.
*   `spark.table()` → **Reads** an existing table into a DataFrame.

***

#### **Common Gotcha**

If you don’t see the data after writing:

*   Check if you are in the correct **catalog and database** (`workspace.default`).
*   Ensure the underlying format supports overwrite (Delta/Parquet works fine).
*   Verify that the Spark session has access to the metastore.

***



In [0]:

# we can use variables as well , 
#variables will help us to make code Dynamic
df_python_2 = (spark.read.format("csv")
             .option("header","true")
             .load("/Volumes/pyspark_practice/default/files/Practice/BigMart Sales.csv"))

v_catlog = 'workspace'
v_database = 'default'
v_table = 'BigMart_sales_2'
df_python_2.write.mode('overwrite').saveAsTable(f'{v_catlog}.{v_database}.{v_table}')
result =spark.table(f'{v_catlog}.{v_database}.{v_table}')
result.display(5)


## Ingesting CSV  Files with COPY INTO

Using the same set of Parquet files as before, let's use COPY INTO to create our Bronze table again.

We will look at two examples:

1. Examplp 1: Common Schema Mismatch Error

2. Example 2: Preemptively Handling Schema Evolution

I have created one cSV file and did simple upload.
created a new schema , created a new volum , create a folder in it and upload file from local machine.

In [0]:
%sql
use catalog pyspark_practice ;
use timepass ;


In [0]:
%sql
Select * from read_files('/Volumes/pyspark_practice/timepass/practice_files' , Format => "CSV");
-- above code will fetch for all teh files which are csv 

Select * from read_files('/Volumes/pyspark_practice/timepass/practice_files/demo_csv.csv' , Format => "CSV")

-- rescued data column  we will look into it later !

In [0]:
%sql
Drop table if exists demo_csv ;

Create table demo_csv(
  name String ,
  roll_number Double ,
  total_percentage String ,
  _rescued_data String  
);


Copy into  demo_csv
from '/Volumes/pyspark_practice/timepass/practice_files'fileformat  = csv
;

In [0]:
%sql
Drop table if exists demo_csv ;

Create table demo_csv(
  name String ,
  roll_number String ,
  total_percentage String
 --, _rescued_data String  
);


Copy into  demo_csv
from '/Volumes/pyspark_practice/timepass/practice_files'fileformat  = csv
;

The error occurs because the **incoming CSV files have no headers**, so Spark assigns default column names (`_c0`, `_c1`, `_c2`). Your target Delta table expects `name`, `roll_number`, and `total_percentage`, which causes a schema mismatch.

***

### ✅ Why this happens

*   `COPY INTO` reads raw files and infers schema.
*   If the CSV files don’t have headers, the inferred schema uses `_c0`, `_c1`, `_c2`.
*   Your Delta table has different column names, so Databricks throws a mismatch error.

***

### ✅ How to fix it

#### **Option 1: Tell COPY INTO that the CSV has headers**

Add `FORMAT_OPTIONS`:

```sql
COPY INTO demo_csv
FROM '/Volumes/pyspark_practice/timepass/practice_files'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');
```

***

#### **Option 2: If CSV has no headers**

Map columns explicitly using `SELECT`:

```sql
COPY INTO demo_csv
FROM (
  SELECT _c0 AS name, _c1 AS roll_number, _c2 AS total_percentage
  FROM read_files('/Volumes/pyspark_practice/timepass/practice_files', FORMAT => 'CSV')
)
FILEFORMAT = CSV;
```

***

#### **Option 3: Enable schema evolution**

If new columns appear later:

```sql
COPY INTO demo_csv
FROM '/Volumes/pyspark_practice/timepass/practice_files'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');
```

***

### ✅ Quick Check

Run:

```sql
SELECT * FROM read_files('/Volumes/pyspark_practice/timepass/practice_files', FORMAT => 'CSV') LIMIT 5;
```

to confirm whether headers exist.

***

Do you want me to give you:

*   ✅ A **full working COPY INTO script for header and no-header cases**?
*   ✅ Or a **best-practice workflow for loading CSV into Delta with schema evolution**?


In [0]:
%sql
Drop table if exists demo_csv ;

Create table demo_csv(
  name String ,
  roll_number String ,
  total_percentage String
 , _rescued_data String  
);

COPY INTO demo_csv
FROM '/Volumes/pyspark_practice/timepass/practice_files'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');


This error is coming from **Delta Lake’s column name validation**. Delta does not allow certain characters (spaces, commas, semicolons, etc.) in column names unless you enable **column mapping**.

***

### ✅ Why this happened

Your table definition:

```sql
Create table demo_csv(
  name String ,
  roll_number String ,
  total_percentage String
 , _rescued_data String  
);
```

looks fine, but the error suggests the actual column names being processed have **leading/trailing spaces** or invalid characters. For example:

    " roll_number "  (with spaces)
    " total_percentage " (with spaces)

***

### ✅ How to fix it

#### **Option 1: Clean column names**

Trim spaces and rename columns before writing:

```sql
COPY INTO demo_csv
FROM (
  SELECT `name` AS name,
         TRIM(`roll_number`) AS roll_number,
         TRIM(`total_percentage`) AS total_percentage
  FROM read_files('/Volumes/pyspark_practice/timepass/practice_files', FORMAT => 'CSV')
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true');
```

***

#### **Option 2: Enable Column Mapping**

If you want to keep unusual names:

```sql
ALTER TABLE demo_csv SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');
```

Then Delta will allow special characters.

***

#### **Option 3: Rename columns in the source**

If you control the CSV, remove spaces in headers:

    name,roll_number,total_percentage

***

### ✅ Best Practice

Always normalize column names before creating Delta tables:

*   Lowercase
*   Replace spaces with `_`
*   Avoid special characters

***

Do you want me to give you:

*   ✅ A **full working COPY INTO script with column renaming and schema evolution**?
*   ✅ Or a **quick PySpark snippet to clean column names automatically before writing to Delta**?
